| Column   | Unique Values | Notes  | **Recommended Encoding**                                       Why                                                            |
| ------------------------- | ------------- | ----------------------------------------------------------------------- | ----------------------------------------------------------------- | -------------------------------------------------------------- |
| **waterpoint_type_group** | 6             | Small, nominal                                                          | **One-Hot Encoding**                                              | Keeps categories distinct without implying order               |
| **management_customize**  | 7             | Small, but may correlate strongly with target (community vs government) | **CatBoost Encoder** *(or TargetEncoder if CatBoost unavailable)* | Captures category → target relationship without leakage        |
| **quantity**              | 5             | This is **ordered**: *enough > insufficient > dry*, etc.                | **Ordinal Encoding** *(manual order)*                             | Ordinal makes sense here because there is a meaningful ranking |
| **source_type**           | 7             | Nominal, no natural order                                               | **One-Hot Encoding**                                              | Clear separation, low category count                           |
| **quality_group**         | 6             | Nominal                                                                 | **One-Hot Encoding**                                              | Values don’t represent order or magnitude                      |
| **region**                | 21            | **High-cardinality geographic** variable                                | **CatBoost Encoder**                                              | Reduces dimensionality and captures location effect            |
| **basin**                 | 9             | Categorical, but geography influences failure                           | **CatBoost Encoder**                                              | Geographic group encodes well using target/catboost style      |


In [ ]:
| Column   | Unique Values | Notes          **Recommended Encoding**                                          | Why                                                            |
| ------------------------- | ------------- | ----------------------------------------------------------------------- | ----------------------------------------------------------------- | -------------------------------------------------------------- |
| **waterpoint_type_group** | 6             | Small, nominal                                                          | **One-Hot Encoding**                                              | Keeps categories distinct without implying order               |
| **management_customize**  | 7             | Small, but may correlate strongly with target (community vs government) | **CatBoost Encoder** *(or TargetEncoder if CatBoost unavailable)* | Captures category → target relationship without leakage        |
| **quantity**              | 5             | This is **ordered**: *enough > insufficient > dry*, etc.                | **Ordinal Encoding** *(manual order)*                             | Ordinal makes sense here because there is a meaningful ranking |
| **source_type**           | 7             | Nominal, no natural order                                               | **One-Hot Encoding**                                              | Clear separation, low category count                           |
| **quality_group**         | 6             | Nominal                                                                 | **One-Hot Encoding**                                              | Values don’t represent order or magnitude                      |
| **region**                | 21            | **High-cardinality geographic** variable                                | **CatBoost Encoder**                                              | Reduces dimensionality and captures location effect            |
| **basin**                 | 9             | Categorical, but geography influences failure                           | **CatBoost Encoder**                                              | Geographic group encodes well using target/catboost style      |


In [ ]:
def norm(s):
    if pd.isna(s): return ""
    return str(s).strip().lower()

def annotate(installer, subvillage, ward, loose=False):
    i, s, w = norm(installer), norm(subvillage), norm(ward)
    if not i:
        return installer
    if loose:
        if s and (s in i or i in s):
            return f"{installer} ({subvillage.strip()} Subvillage)" if str(subvillage).strip() else f"{installer} (Subvillage)"
        if w and (w in i or i in w):
            return f"{installer} ({ward.strip()} Ward)" if str(ward).strip() else f"{installer} (Ward)"
    else:
        if s and i == s:
            return f"{installer} ({subvillage.strip()} Subvillage)" if str(subvillage).strip() else f"{installer} (Subvillage)"
        if w and i == w:
            return f"{installer} ({ward.strip()} Ward)" if str(ward).strip() else f"{installer} (Ward)"
    return installer

# Example on a DataFrame df with columns: 'installer','subvillage','ward'
# df["installer_annotated"] = df.apply(lambda r: annotate(r["installer"], r["subvillage"], r["ward"], loose=False), axis=1)

In [ ]:
import re
import pandas as pd
from typing import Optional, Tuple

def _normalize(s: Optional[str]) -> str:
    if not isinstance(s, str): return ""
    s = s.lower().strip()
    s = re.sub(r"\s+", " ", s)             # collapse spaces
    s = re.sub(r"[^a-z0-9\s/+-]", "", s)   # keep letters/digits/space and a few safe symbols
    return s

def annotate_installer_if_subvillage(
    df: pd.DataFrame,
    installer_col: str = "installer",
    subvillage_col: str = "subvillage",
    *,
    new_flag_col: str = "installer_is_subvillage",
    new_text_col: str = "installer_with_sv",
    strategy: str = "normalized_exact",    # "exact" | "normalized_exact"
    tag: str = "(subvillage)"
) -> Tuple[pd.Series, pd.Series]:
    """
    If installer matches a subvillage value, append '(subvillage)' to the installer text.
    Returns (flag_series, annotated_text_series). Also writes both columns to df.
    """
    s_inst = df[installer_col].astype("string")
    s_subv = df[subvillage_col].astype("string")

    if strategy == "exact":
        sv_set = set(s_subv.dropna().unique())
        flag = s_inst.isin(sv_set)
    elif strategy == "normalized_exact":
        sv_set = set(_normalize(v) for v in s_subv.dropna().unique())
        flag = s_inst.apply(lambda x: _normalize(x) in sv_set)
    else:
        raise ValueError("strategy must be 'exact' or 'normalized_exact'.")

    # prevent double-tagging
    already_tagged = s_inst.fillna("").str.contains(re.escape(tag), na=False)

    annotated = s_inst.where(~(flag & ~already_tagged), s_inst.fillna("") + tag)

    df[new_flag_col] = flag
    df[new_text_col] = annotated

    return df[new_flag_col], df[new_text_col]

In [ ]:
flag, annotated = annotate_installer_if_subvillage(
    train_df,
    installer_col="installer",
    subvillage_col="subvillage",
    new_flag_col="installer_is_subvillage",
    new_text_col="installer_with_sv",
    strategy="normalized_exact",   # safest default
    tag="(subvillage)"             # matches your format: no space before bracket
)

# Quick checks
flag.sum(), train_df.loc[flag, ["installer","subvillage","installer_with_sv"]].head(10)
train_df["installer_with_sv"]


In [ ]:
# --- CLUSTER installer_with_sv WITH TAG-AWARE LOGIC ---

import re
import pandas as pd
from typing import Callable, Dict, List, Tuple, Iterable, Optional
try:
    from rapidfuzz import process, fuzz
except ImportError:
    raise ImportError("Please install rapidfuzz: pip install rapidfuzz")

TAG = "(subvillage)"
TAG_RE = re.compile(r"\s*\(subvillage\)$", re.IGNORECASE)

def _norm_basic(s: Optional[str]) -> str:
    if not isinstance(s, str): return ""
    s = s.lower().strip()
    s = re.sub(r"\s+\(subvillage\)$", "", s)  # remove tag ONLY for normalization (not for output)
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^a-z0-9\s/+-]", "", s)      # keep simple chars
    return s

def _has_tag(s: Optional[str]) -> bool:
    return bool(isinstance(s, str) and TAG_RE.search(s))

def _build_fuzzy_clusters(
    values: Iterable[str] | pd.Series,
    *,
    threshold: int = 88,
    scorer = fuzz.token_sort_ratio,
    normalize_fn: Optional[Callable[[str], str]] = _norm_basic,
    canon_strategy: str = "most_common"  # "most_common" | "shortest" | "first"
) -> Tuple[Dict[str, List[str]], Dict[str, str]]:
    """Generic clusterer (single pool)."""
    s = pd.Series(list(values), dtype="string")
    counts = s.value_counts(dropna=False)
    uniq = pd.Index(s.dropna().unique()).astype(str).tolist()

    norm_map = {u: normalize_fn(u) for u in uniq} if normalize_fn else {u: u for u in uniq}
    remaining = set(norm_map[u] for u in uniq)

    norm_to_orig: Dict[str, List[str]] = {}
    for orig, normed in norm_map.items():
        norm_to_orig.setdefault(normed, []).append(orig)

    clusters: Dict[str, List[str]] = {}
    v2c: Dict[str, str] = {}

    def pick_canonical(members: List[str]) -> str:
        if canon_strategy == "shortest":
            return min(members, key=lambda x: (len(x), x))
        if canon_strategy == "first":
            return members[0]
        return max(members, key=lambda x: (counts.get(x, 0), -len(x)))  # most common, tie -> shorter

    while remaining:
        seed_norm = remaining.pop()
        pool = list(remaining) + [seed_norm]
        matches = process.extract(seed_norm, pool, scorer=scorer, limit=None)
        group_norm = {seed_norm} | {m for (m, score, _) in matches if score >= threshold}
        remaining -= (group_norm - {seed_norm})

        members: List[str] = []
        for n in group_norm:
            members.extend(norm_to_orig.get(n, []))
        members = sorted(set(members))

        canonical = pick_canonical(members)
        clusters[canonical] = [m for m in members if m != canonical]
        for m in members:
            v2c[m] = canonical

    return clusters, v2c

def cluster_installer_with_sv(
    df: pd.DataFrame,
    col: str = "installer_with_sv",
    *,
    threshold_tagged: int = 90,    # a bit stricter for tagged side
    threshold_untagged: int = 88,  # standard for untagged
    scorer = fuzz.token_sort_ratio,
) -> Tuple[Dict[str, List[str]], Dict[str, str]]:
    s = df[col].astype("string")

    tagged = s[s.apply(_has_tag)]
    untagged = s[~s.apply(_has_tag)]

    clusters_tag, v2c_tag = _build_fuzzy_clusters(
        tagged, threshold=threshold_tagged, scorer=scorer, normalize_fn=_norm_basic
    )
    clusters_untag, v2c_untag = _build_fuzzy_clusters(
        untagged, threshold=threshold_untagged, scorer=scorer, normalize_fn=_norm_basic
    )

    # merge maps + clusters
    clusters = {**clusters_tag, **clusters_untag}
    v2c = {**v2c_tag, **v2c_untag}

    # apply canonical column (tag is preserved because we never stripped it from originals)
    df[col + "_canon"] = s.map(v2c).fillna(s)

    return clusters, v2c

def export_cluster_views(
    clusters: Dict[str, List[str]],
    path_detailed: str = "installer_with_sv_clusters_detailed.csv",
    path_summary: str = "installer_with_sv_clusters_summary.csv",
) -> Tuple[str, str]:
    rows = []
    for canon, variants in clusters.items():
        for v in sorted(set([canon] + variants)):
            rows.append((canon, v))
    detailed = (
        pd.DataFrame(rows, columns=["canonical", "variant"])
          .value_counts(["canonical","variant"])
          .rename("count")
          .reset_index()
          .sort_values(["canonical","count"], ascending=[True, False])
    )
    summary = (detailed.groupby("canonical")["count"]
               .agg(total_count="sum")
               .reset_index()
               .assign(n_variants=detailed.groupby("canonical")["variant"].nunique().values)
               .sort_values("total_count", ascending=False))

    detailed.to_csv(path_detailed, index=False)
    summary.to_csv(path_summary, index=False)
    return path_detailed, path_summary


In [ ]:
COL = "installer_with_sv"  # your tagged column

clusters, v2c = cluster_installer_with_sv(
    train_df, col=COL,
    threshold_tagged=90,
    threshold_untagged=88,
    scorer=fuzz.token_sort_ratio
)

# Export for Excel review
p_det, p_sum = export_cluster_views(clusters)
print("Wrote:")
print(" -", p_det)
print(" -", p_sum)

# Quick sanity checks
#print(train_df[COL + "_canon"].value_counts().head(20))


In [ ]:
import pandas as pd, pathlib, sys, os, subprocess

detailed = pd.read_csv("installer_with_sv_clusters_detailed.csv")
summary  = pd.read_csv("installer_with_sv_clusters_summary.csv")

xlsx_path = pathlib.Path("installer_with_sv_clusters.xlsx").resolve()
with pd.ExcelWriter(xlsx_path, engine="xlsxwriter") as xw:
    detailed.to_excel(xw, index=False, sheet_name="detailed")
    summary.to_excel(xw,  index=False, sheet_name="summary")

print("Saved:", xlsx_path)

# Open it
if sys.platform.startswith("win"):
    os.startfile(str(xlsx_path))
elif sys.platform == "darwin":
    subprocess.run(["open", str(xlsx_path)])
else:
    subprocess.run(["xdg-open", str(xlsx_path)])


In [ ]:
def grouped_cluster_apply(
    df: pd.DataFrame,
    col: str,
    group_by: Sequence[str],
    *,
    threshold: int = 90,
    scorer=fuzz.token_sort_ratio,
    normalize_fn=norm_place,
    dst_col: Optional[str] = None
) -> Tuple[Dict[Tuple, Dict[str, List[str]]], Dict[Tuple, Dict[str, str]]]:
    """
    Cluster `col` within each group defined by `group_by` (e.g., region, ward).
    Returns dicts keyed by group tuple: {group_key: clusters}, {group_key: v2c}
    Optionally writes canonicalized column dst_col (default: f"{col}_canon").
    """
    if dst_col is None:
        dst_col = f"{col}_canon"
    df[dst_col] = df[col].astype("string")  # initialize

    clusters_all = {}
    v2c_all = {}

    # group keys always tuples for consistency
    gb = df.groupby(list(group_by), dropna=False, sort=False)
    for gkey, gdf in gb:
        vals = gdf[col].dropna().astype("string").unique().tolist()
        clusters, v2c = build_fuzzy_clusters(
            vals, threshold=threshold, scorer=scorer, normalize_fn=normalize_fn
        )
        clusters_all[gkey] = clusters
        v2c_all[gkey] = v2c

        if v2c:
            idx = gdf.index
            df.loc[idx, dst_col] = df.loc[idx, col].map(v2c).fillna(df.loc[idx, col])

    return clusters_all, v2c_all

def export_grouped_clusters(
    clusters_by_group: Dict[Tuple, Dict[str, List[str]]],
    *,
    detailed_path: str,
    summary_path: str
) -> Tuple[str, str]:
    rows = []
    for gkey, clusters in clusters_by_group.items():
        gkey = gkey if isinstance(gkey, tuple) else (gkey,)
        for canon, variants in clusters.items():
            for v in sorted(set([canon] + variants)):
                rows.append((*gkey, canon, v))

    if not rows:
        pd.DataFrame(columns=[*map(str, range(999)), "canonical", "variant"]).to_csv(detailed_path, index=False)
        pd.DataFrame(columns=["canonical", "total_count", "n_variants"]).to_csv(summary_path, index=False)
        return detailed_path, summary_path

    # infer group column names like group_0, group_1...
    num_gcols = len([r for r in rows[0]]) - 2
    gcols = [f"group_{i}" for i in range(num_gcols)]
    detailed = pd.DataFrame(rows, columns=[*gcols, "canonical", "variant"])
    # collapse duplicates (if any) and count
    detailed = (detailed
        .value_counts([*gcols, "canonical", "variant"])
        .rename("n_values")
        .reset_index()
        .sort_values([*gcols, "canonical", "n_values"], ascending=[True]*len(gcols) + [True, False])
    )
    summary = (detailed
        .groupby([*gcols, "canonical"], as_index=False)["n_values"].sum()
        .rename(columns={"n_values":"total_count"})
    )
    # variants per canonical
    nvar = detailed.groupby([*gcols, "canonical"])["variant"].nunique().reset_index(name="n_variants")
    summary = summary.merge(nvar, on=[*gcols, "canonical"]).sort_values([*gcols, "total_count"], ascending=[True]*len(gcols)+[False])

    detailed.to_csv(detailed_path, index=False)
    summary.to_csv(summary_path, index=False)
    return detailed_path, summary_path


In [ ]:
# Cluster ward inside each region
ward_clusters, ward_v2c = grouped_cluster_apply(
    train_df,
    col="ward",
    group_by=["region"],        # or ["lga"] if that’s cleaner
    threshold=90,               # wards tend to be similar; keep it moderately strict
    scorer=fuzz.token_sort_ratio,
    normalize_fn=norm_place,
    dst_col="ward_canon"
)

# Export for Excel
export_grouped_clusters(
    ward_clusters,
    detailed_path="ward_clusters_by_region_detailed.csv",
    summary_path="ward_clusters_by_region_summary.csv"
)

Scaling is now applied to both the raw numeric and the log-transformed branch.

Everything (imputation, scaling, encoders) fits inside each CV fold.

Output is dense; if memory becomes an issue, switch OneHotEncoder(sparse=True) and remove sparse_threshold=0.0 (and use models that accept sparse matrices).

| Encoding Method                     | Columns                                                     |
| ----------------------------------- | ----------------------------------------------------------- |
| **One-Hot Encoding**                | `waterpoint_type_group`, `source_type`, `quality_group`     |
| **Ordinal Encoding (manual order)** | `quantity`                                                  |
| **CatBoost Encoding**               | `management_customize`, `region`, `basin`, `installer_cate` |

In [ ]:
python -m pip install category_encoders ..Run installs like this (forces the right pip):
ohe_cols = ['waterpoint_type_group', 'source_type', 'quality_group']
ordinal_cols = ['quantity']
targetenc_cols = [
   'management_customize', 'installer_cate', 'region', 'basin',
   'management_payment_combo', 'source_waterpoint_combo'
]

from installer_dict import installer_groups  

In [ ]:
# catboost 
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer
from catboost import CatBoostClassifier
import numpy as np

# Use your existing column lists:
# numeric_cols, log_cols
# ohe_cols, ordinal_cols, targetenc_cols  -> we'll treat these as *raw categoricals* for CatBoost
cat_cols_for_cb = ohe_cols + ordinal_cols + targetenc_cols

# Preprocess for CatBoost: impute numerics + log1p branch; pass raw categoricals (imputed) through
num_imputer_cb = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
])

log_branch_cb = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('log1p', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
])

cat_imputer_cb = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
])

preprocess_catboost = ColumnTransformer(
    transformers=[
        ('num', num_imputer_cb, numeric_cols),
        ('log', log_branch_cb, log_cols),
        ('cat', cat_imputer_cb, cat_cols_for_cb),  # raw categories, no OHE
    ],
    remainder='drop',
    sparse_threshold=0.0
)

# The transformed matrix will be: [num | log | cat]
n_num, n_log = len(numeric_cols), len(log_cols)
cat_features_idx = list(range(n_num + n_log, n_num + n_log + len(cat_cols_for_cb)))


cat_model = CatBoostClassifier(
    loss_function="MultiClass",
    auto_class_weights="Balanced",  # handles imbalance
    depth=8,
    learning_rate=0.1,
    iterations=500,
    random_state=42,
    verbose=0
)

pipe_catboost = Pipeline(steps=[
    ("preprocess", preprocess_catboost),
    ("clf", cat_model),
])

# Add to your model dictionary
all_pipes["CatBoost (baseline)"] = pipe_catboost

# Map of fit_params per pipeline name
fit_params_map = {
    "CatBoost (baseline)": {"clf__cat_features": cat_features_idx}
}

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, f1_score, balanced_accuracy_score, precision_score, recall_score, log_loss

def run_cv_for_pipes(pipes, X, y, n_splits=5, random_state=42, fit_params_map=None):
    if fit_params_map is None:
        fit_params_map = {}
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    rows = []
    for name, pipe in pipes.items():
        est = pipe.named_steps["clf"]
        scoring = {
            "accuracy": "accuracy",
            "bal_acc": make_scorer(balanced_accuracy_score),
            "f1_macro": make_scorer(f1_score, average="macro"),
            "f1_weighted": make_scorer(f1_score, average="weighted"),
            "prec_macro": make_scorer(precision_score, average="macro", zero_division=0),
            "prec_weighted": make_scorer(precision_score, average="weighted", zero_division=0),
            "recall_macro": make_scorer(recall_score, average="macro", zero_division=0),
            "recall_weighted": make_scorer(recall_score, average="weighted", zero_division=0),
        }
        # log_loss only if proba available (CatBoost has it)
        if hasattr(est, "predict_proba"):
            scoring["neg_log_loss"] = make_scorer(log_loss, greater_is_better=False, needs_proba=True)

        fp = fit_params_map.get(name, {})
        out = cross_validate(pipe, X, y, cv=skf, scoring=scoring, n_jobs=-1, return_train_score=False, fit_params=fp)

        row = {"model": name}
        for k, v in out.items():
            if not k.startswith("test_"): 
                continue
            metric = k.replace("test_", "")
            mean, std = float(np.mean(v)), float(np.std(v))
            if metric == "neg_log_loss":
                metric, mean = "log_loss", -mean
            row[f"{metric}_mean"] = mean
            row[f"{metric}_std"] = std
        rows.append(row)
    df = (pd.DataFrame(rows).set_index("model")
          .sort_values("f1_macro_mean", ascending=False))
    return df

def evaluate_holdout(pipe, X_train, y_train, X_val, y_val, fit_params=None):
    if fit_params is None:
        fit_params = {}
    pipe.fit(X_train, y_train, **fit_params)
    # ... (same as before: predict, metrics, confusion matrix)


results_df = run_cv_for_pipes(all_pipes, X_train, y_train, n_splits=5, random_state=42, fit_params_map=fit_params_map)
best_name = results_df.index[0]
best_pipe = all_pipes[best_name]
best_fit_params = fit_params_map.get(best_name, {})
metrics_val, cm_val, labels_val, report_val = evaluate_holdout(best_pipe, X_train, y_train, X_val, y_val, fit_params=best_fit_params)


3) What to look for

Macro-F1 vs Weighted-F1: big gaps suggest minority classes are struggling.

Row-normalized CM: low diagonal entries → poor recall for those classes.

Column-normalized CM: low diagonal entries → poor precision for those classes.

Compare the OOF view to the hold-out view to detect overfitting.

That covers both: comparison tables with multiple metrics and a clear analysis across classes. If you want, we can add quick plots (normalized CM heatmaps) next.

In [ ]:
Strengths vs. Weaknesses (and interpretability vs. performance)

Logistic Regression (multinomial)

Strengths: Interpretable (signs/magnitudes of coefficients), fast to train/infer, stable, works well with many sparse OHE features, calibrated probabilities.

Weaknesses: Linear decision boundary; can underperform on non-linear interactions unless you craft features.

Imbalance: Use class_weight='balanced'; SMOTE sometimes helps but less critical for linear models.

Interpretability: ⭐⭐⭐⭐⭐ | Typical performance: ⭐⭐–⭐⭐⭐

LinearSVC (+ calibration)

Strengths: Strong linear margin; good on high-dimensional sparse data; with calibration gives usable probabilities.

Weaknesses: Still linear; calibration adds cost; fewer direct feature importances.

Imbalance: class_weight='balanced' helps.

Interpretability: ⭐⭐⭐ | Perf: ⭐⭐–⭐⭐⭐

RBF SVC

Strengths: Very strong non-linear boundaries on small/medium data; often lifts minority class recall with SMOTE.

Weaknesses: Slower training/inference; sensitive to C/γ; scaling required.

Imbalance: Responds well to SMOTE; class weights can help too.

Interpretability: ⭐⭐ | Perf: ⭐⭐⭐–⭐⭐⭐⭐

Decision Tree

Strengths: Human-readable rules; captures non-linear interactions; no scaling needed.

Weaknesses: High variance/overfitting; single tree usually weaker than ensembles.

Imbalance: Use class_weight='balanced' or prune; SMOTE can help.

Interpretability: ⭐⭐⭐⭐ | Perf: ⭐⭐

Random Forest / ExtraTrees

Strengths: Strong baselines; robust; handle non-linearities; low tuning burden; feature importances.

Weaknesses: Less interpretable than single tree; probabilities can be a bit overconfident; inference slower than linear.

Imbalance: class_weight='balanced' often enough; try SMOTE vs no-SMOTE.

Interpretability: ⭐⭐⭐ | Perf: ⭐⭐⭐–⭐⭐⭐⭐

Gradient Boosting / LightGBM / CatBoost

Strengths: Often best on tabular data; capture complex interactions; CatBoost handles categoricals natively; good calibrated probs.

Weaknesses: More hyperparameters; risk of overfitting without early stopping; requires careful CV.

Imbalance: Prefer class weights / built-ins (e.g., CatBoost auto_class_weights); compare to SMOTE.

Interpretability: ⭐⭐⭐ (global SHAP/feature importance) | Perf: ⭐⭐⭐⭐–⭐⭐⭐⭐⭐

KNN (for completeness)

Strengths: Simple, non-parametric.

Weaknesses: Slow at inference; struggles in high-dimensional OHE; sensitive to scaling and k.

Interpretability: ⭐⭐ | Perf: ⭐–⭐⭐

SMOTE vs. Class Weights (general)

SMOTE: Can lift recall for minority classes, especially with SVC/KNN; risk of overfitting if the minority class is tiny/noisy; must be inside CV folds.

Class Weights: Cheap, simple, often enough for linear and tree ensembles; usually my default first try.

Decision Matrix for Final Model Selection
Suggested criteria & weights (adjust to taste)

Macro-F1 (CV): 0.35 — primary metric under imbalance

Balanced Accuracy (CV): 0.25 — class-balanced hit rate

Log Loss (CV, if available): 0.10 — probability quality

Interpretability: 0.15 — ease of explaining decisions

Inference Speed: 0.10 — throughput, latency

Maintenance/Complexity: 0.05 — tuning & ops burden

How to build it from your results

Start from your results_df (we already created it) for the CV metrics.

Add subjective scores (1–5) for interpretability, inference speed, and maintenance by model family.

Normalize CV metrics so higher is better (invert log loss).

Compute a weighted sum and rank.

Here’s a minimal snippet you can drop in a new cell:

In [ ]:
import pandas as pd
import numpy as np

# 1) pick the columns we have
dm = results_df.copy()  # rows = model names
# ensure missing metrics exist
for col in ["f1_macro_mean","bal_acc_mean","log_loss_mean"]:
    if col not in dm.columns:
        dm[col] = np.nan

# 2) subjective scores by family (1–5). Adjust if your constraints differ.
def subjective_scores(model_name):
    name = model_name.lower()
    if "logreg" in name:
        return dict(interpret=5, speed=5, maintain=5)
    if "linearsvc" in name:
        return dict(interpret=3, speed=4, maintain=4)
    if "svc" in name:  # rbf
        return dict(interpret=2, speed=2, maintain=3)
    if "decisiontree" in name:
        return dict(interpret=4, speed=4, maintain=4)
    if "randomforest" in name or "extratrees" in name:
        return dict(interpret=3, speed=3, maintain=4)
    if "catboost" in name or "lightgbm" in name or "gradientboosting" in name:
        return dict(interpret=3, speed=3, maintain=3)
    if "knn" in name:
        return dict(interpret=2, speed=1, maintain=4)
    # default
    return dict(interpret=3, speed=3, maintain=3)

subj = dm.index.to_series().apply(subjective_scores).apply(pd.Series)
dm = pd.concat([dm, subj], axis=1)

# 3) normalize metrics to [0,1] (higher better). For log_loss, invert.
def minmax(s):
    s = s.copy()
    if s.isna().all():
        return s
    return (s - s.min()) / (s.max() - s.min() + 1e-12)

dm["macro_f1_n"] = minmax(dm["f1_macro_mean"])
dm["bal_acc_n"] = minmax(dm["bal_acc_mean"])

# if some models don't have log_loss, fill with median so they aren't unfairly penalized
ll = dm["log_loss_mean"]
ll_filled = ll.fillna(ll.median())
dm["log_loss_n"] = 1 - minmax(ll_filled)  # lower log loss => higher score

# subjective are already 1..5; scale to 0..1
for c in ["interpret", "speed", "maintain"]:
    dm[c+"_n"] = (dm[c] - 1) / 4.0

# 4) weights & overall score
w = dict(macro_f1=0.35, bal_acc=0.25, log_loss=0.10, interpret=0.15, speed=0.10, maintain=0.05)
dm["overall"] = (
    w["macro_f1"]*dm["macro_f1_n"] +
    w["bal_acc"]*dm["bal_acc_n"] +
    w["log_loss"]*dm["log_loss_n"] +
    w["interpret"]*dm["interpret_n"] +
    w["speed"]*dm["speed_n"] +
    w["maintain"]*dm["maintain_n"]
)

decision_table = dm[[
    "f1_macro_mean","bal_acc_mean","log_loss_mean",
    "interpret","speed","maintain","overall"
]].sort_values("overall", ascending=False)

print(decision_table.round(4).head(10))


In [ ]:
Takeaways to document

Interpretability vs. performance:

Linear models are easiest to explain; boosted trees often win on accuracy.

RF/ET are a great “middle ground” (strong, reasonably interpretable via feature importances or SHAP).

CatBoost/LightGBM typically top performance with modest complexity; interpretability via SHAP is good enough for most stakeholders.

Imbalance strategy:

Start with class weights; add SMOTE where it actually improves macro-F1/recall (commonly SVC/KNN).

Final selection:

Use the decision matrix to justify the choice against business constraints, not just raw CV scores.

In [ ]:
# ===== Preprocessing Pipeline =====
# Define column types
numeric_cols = ['age', 'construction_decade', 'payment_score']
log_cols = ['population_clean', 'amount_tsh_nonzero']

# explicit encoder groups
ohe_cols = ['waterpoint_type_group', 'source_type', 'quality_group']
ordinal_cols = ['quantity']
targetenc_cols = ['management_customize', 'region', 'basin','management_payment_combo', 'source_waterpoint_combo']

# order for the ordinal feature
quantity_order = [['dry', 'insufficient', 'seasonal', 'enough', 'unknown']]


# ===== preprocessing blocks =====
num_imputer = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler())
])

log_branch = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='median')),
    ('log1p', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
    ('scale', StandardScaler())
])

# NOTE: for sklearn <1.2, replace sparse_output=False with sparse=False
ohe_pipe = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

ordinal_pipe = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('ord', OrdinalEncoder(
        categories=quantity_order,
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

# TargetEncoder will receive y from the outer Pipeline fit(X, y) — safe for CV
targetenc_pipe = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('te', TargetEncoder(smoothing=0.3))
])

preprocess = ColumnTransformer(
    transformers=[
        ('num', num_imputer, numeric_cols), ('log', log_branch, log_cols),
        ('ohe', ohe_pipe,ohe_cols), ('ord', ordinal_pipe, ordinal_cols),
        ('te', targetenc_pipe, targetenc_cols),
    ],
    remainder='drop',
    sparse_threshold=0.0   # force dense output (good for SMOTE + StandardScaler)
)


In [ ]:
# pipeline preprocessing
def _onehot(sparse_output=True):
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=sparse_output)
    except TypeError:
        # fallback for older sklearn where 'sparse_output' isn't available
        return OneHotEncoder(handle_unknown="ignore", sparse=sparse_output)

def make_preprocess(
    numeric_cols, log_cols, ohe_cols, ordinal_cols, targetenc_cols, quantity_order, *, te_smoothing=0.3,
    do_scale=True, force_dense=True
):
    
    num_steps = [("impute", SimpleImputer(strategy="median"))]
    if do_scale:
        num_steps.append(("scale", StandardScaler()))
    num_imputer = Pipeline(steps=num_steps)

    log_steps = [
        ("impute", SimpleImputer(strategy="median")),
        ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ]
    if do_scale:
        log_steps.append(("scale", StandardScaler()))
    log_branch = Pipeline(steps=log_steps)

    ohe_pipe = Pipeline(steps=[
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("ohe", _onehot(sparse_output=True))
    ])

    ordinal_pipe = Pipeline(steps=[
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("ord", OrdinalEncoder(
            categories=quantity_order,
            handle_unknown="use_encoded_value",
            unknown_value=-1
        ))
    ])

    targetenc_pipe = Pipeline(steps=[
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("te", TargetEncoder(smoothing=te_smoothing))
    ])

    preprocess = ColumnTransformer(
        transformers=[
            ("num",  num_imputer,  numeric_cols),
            ("log",  log_branch,   log_cols),
            ("ohe",  ohe_pipe,     ohe_cols),
            ("ord",  ordinal_pipe, ordinal_cols),
            ("te",   targetenc_pipe, targetenc_cols),
        ],
        remainder="drop",
        sparse_threshold=(0.0 if force_dense else 0.3)
    )
    return preprocess

preprocess = make_preprocess(numeric_cols=numeric_cols, log_cols=log_cols, ohe_cols=ohe_cols,
    ordinal_cols=ordinal_cols, targetenc_cols=targetenc_cols, 
    quantity_order=[['dry','insufficient','seasonal','enough','unknown']], te_smoothing=0.3, do_scale=True,
    force_dense=True   
)

In [ ]:
# Top-N comparison table (means only)
cmp_cols = [c for c in results_df.columns if c.endswith("_mean")]
display(results_df[cmp_cols].round(4).head(10))

# If you also want ± std in one cell:
def mean_pm_std(df, metric):
    m, s = df[f"{metric}_mean"], df.get(f"{metric}_std")
    return (m.round(4).astype(str) + (" ± " + s.round(4).astype(str) if s is not None else ""))

table = {
    "f1_macro": mean_pm_std(results_df, "f1_macro"),
    "bal_acc": mean_pm_std(results_df, "bal_acc"),
    "f1_weighted": mean_pm_std(results_df, "f1_weighted"),
    "prec_macro": mean_pm_std(results_df, "prec_macro") if "prec_macro_mean" in results_df else None,
    "recall_macro": mean_pm_std(results_df, "recall_macro") if "recall_macro_mean" in results_df else None,
    "accuracy": mean_pm_std(results_df, "accuracy"),
    "log_loss": mean_pm_std(results_df, "log_loss") if "log_loss_mean" in results_df else None,
}
pd.DataFrame({k:v for k,v in table.items() if v is not None}).head(10)

In [ ]:
# pick a model to inspect (e.g., best by macro-F1)
model_name = results_df.index[0]
pipe = all_pipes[model_name]
fit_params = fit_params_map.get(model_name, {}) if 'fit_params_map' in globals() else {}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred_oof = cross_val_predict(pipe, X_train, y_train, cv=skf, method="predict", n_jobs=-1, fit_params=fit_params)

# classification report as dict → DataFrame
rep = classification_report(y_train, y_pred_oof, output_dict=True, digits=4)
per_class_df = pd.DataFrame(rep).T  # rows: classes + averages
per_class_df.loc[[c for c in per_class_df.index if c not in ["accuracy","macro avg","weighted avg"]]].round(4)


In [ ]:
 Hold-out per-class metrics
# Using the model refit on X_train and evaluated on X_val

best_pipe = all_pipes[model_name]
best_pipe.fit(X_train, y_train, **fit_params)
y_val_pred = best_pipe.predict(X_val)

rep_val = classification_report(y_val, y_val_pred, output_dict=True, digits=4)
per_class_val_df = pd.DataFrame(rep_val).T
per_class_val_df.loc[[c for c in per_class_val_df.index if c not in ["accuracy","macro avg","weighted avg"]]].round(4)


In [ ]:
# Out-of-fold confusion matrix (normalized two ways)
# This helps see who confuses with whom:
from sklearn.metrics import confusion_matrix
labels = np.unique(y_train)
cm = confusion_matrix(y_train, y_pred_oof, labels=labels)

# normalize by true class (recall per class)
cm_row_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

# normalize by predicted class (precision perspective)
cm_col_norm = cm.astype(float) / cm.sum(axis=0, keepdims=True)

print("Labels:", list(labels))
print("Confusion matrix (counts):\n", cm)
print("Row-normalized (per-class recall):\n", np.round(cm_row_norm, 3))
print("Column-normalized (precision perspective):\n", np.round(cm_col_norm, 3))


In [ ]:
X = train_df[feature_cols]
y = train_df['status_group']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

pipe.fit(X_train, y_train)
pred = pipe.predict(X_val)

print("Accuracy:", accuracy_score(y_val, pred))
print(classification_report(y_val, pred))


In [ ]:
X = train_df[feature_cols]
y = train_df['status_group']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

pipe.fit(X_train, y_train)
pred = pipe.predict(X_val)

print("Accuracy:", accuracy_score(y_val, pred))
print(classification_report(y_val, pred))


test_ids = test_df['id'].copy()
X_test = test_df[feature_cols]
y_test_pred = pipe.predict(X_test)

submission = pd.DataFrame({'id': test_ids, 'status_group': y_test_pred})


In [ ]:
#If you want a confusion matrix with normalized proportions:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt

cm = confusion_matrix(y_val, y_pred, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=np.unique(y_val))
disp.plot(cmap='Blues', xticks_rotation=45)
plt.title("Normalized Confusion Matrix – Hold-out set")
plt.show()



In [ ]:
# Step 3: Investigate regional error patterns
# Compute accuracy by region
region_perf = (
    val_results.groupby("region")["correct"]
    .agg(["count", "mean"])
    .rename(columns={"count": "samples", "mean": "accuracy"})
)
region_perf["error_rate"] = 1 - region_perf["accuracy"]
region_perf = region_perf.sort_values("error_rate", ascending=False)
print(region_perf.head(10))

In [ ]:
#Step 4: Deep-dive misclassification patterns (optional)
pd.crosstab(val_results["true"], val_results["pred"],
            normalize='index').round(2)

#Step 4: Deep-dive misclassification patterns (optional)
val_results.groupby("pred")[["population","construction_year"]].mean()


In [ ]:
def choose_features_after_split(X_train, X_val, feature_cols):
    X_tr = X_train[feature_cols].copy()
    X_va = X_val[feature_cols].copy()
    return X_tr, X_va

X_train, X_val = choose_features_after_split(X_train, X_val, feature_cols)
